# exp_2 — transfer-robust SetSeq (pre-registered in `PREREG.md`)

**Question.** exp_1's SetSeq wins on the in-domain proxy (0.656 spellF1), but IDSSE parks the ball off
the pitch during dead time (off-pitch on 26 % of visible frames vs 4.2 % in BAR-ATH). Does the model
lean on a dead-ball cue that BAR-ATH does not have, and does *target-like observation augmentation*
fix that without losing in-domain accuracy?

**What is new vs exp_0 / exp_1 / rasya / ryu**

| component | exp_2 | closest prior work |
|---|---|---|
| evaluation | second, label-free **`target_like` view** of the same held-out matches (off-pitch ball hidden after 1.48 s; ball hidden on long slow-play runs) applied to *every* arm | exp_0/exp_1: one in-domain view; ryu: other sources |
| training | **`target_aug`**: two of three corruption rounds lose off-pitch tails (random length) and, in half of the long dead spells, the ball | exp_1: corruption resampling + mirror only |
| decoder (H2) | **explicit-duration HSMM** with entropy-centred log-normal priors from the training folds' labels | exp_1: constant / boundary Viterbi; ryu: rate-calibrated λ |
| ensemble (H3) | 3 training seeds, probabilities averaged | ryu: 3 seeds (TCN) |

**Protocol.** exp_1's frozen judge and nested leave-one-match-out (21 pair + 7 outer fits per arm);
each fit predicts both views with the same weights. Decisions follow `PREREG.md` (hash checked in every
notebook). BAR-ATH is touched only in notebook 05, aggregates only; the public window is never opened.


In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp2_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_2')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
import exp2_common as C
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp2_protocol as P, exp2_views as V
import exp1_common as C1

PREREG verified 154a10f7cda6821e | frozen 2026-11-08T13:06:55+00:00


## 1. Frozen judge, proxy integrity, and what the view changes
The round-0 copies must equal the frozen degrader; the `target_like` view may only remove ball observations.

In [2]:
integrity = pd.DataFrame(P.view_integrity())
C1.write_json(C.RESULTS / 'view_integrity.json', integrity.to_dict(orient='records'))
print(integrity.to_string(index=False))
assert integrity.drop(columns='match').all().all()

 match  frozen_copy_matches_degrader  view_labels_unchanged  view_players_unchanged  view_only_removes_ball
J03WMX                          True                   True                    True                    True
J03WN1                          True                   True                    True                    True
J03WOH                          True                   True                    True                    True
J03WOY                          True                   True                    True                    True
J03WPY                          True                   True                    True                    True
J03WQQ                          True                   True                    True                    True
J03WR9                          True                   True                    True                    True


## 2. Token caches: `target_like` (held-out view) and `target_aug` (training rounds 1-2)
Only IDSSE aggregates. `outside_of_visible` should fall from about 0.26 to about 0.04; labels are used here only to describe the views.

In [3]:
t0 = time.time()
stats = pd.DataFrame(P.build_token_caches())
stats.to_csv(C.RESULTS / 'view_stats.csv', index=False)
cols = ['ball_visible', 'outside_of_visible', 'visible_on_dead', 'visible_on_alive', 'outside_of_visible_dead']
print(stats[['match', 'kind', 'round'] + cols].round(3).to_string(index=False))
print('\nmean by kind'); print(stats.groupby('kind')[cols].mean().round(3).to_string())
print(f'built in {time.time() - t0:.0f}s')

 match        kind  round  ball_visible  outside_of_visible  visible_on_dead  visible_on_alive  outside_of_visible_dead
J03WMX    original      0         0.590               0.334            0.606             0.579                    0.776
J03WMX target_like      0         0.378               0.043            0.099             0.576                    0.359
J03WMX  target_aug      1         0.382               0.037            0.069             0.604                    0.447
J03WMX  target_aug      2         0.373               0.046            0.081             0.580                    0.461
J03WN1    original      0         0.590               0.337            0.596             0.584                    0.639
J03WN1 target_like      0         0.353               0.056            0.138             0.582                    0.247
J03WN1  target_aug      1         0.341               0.043            0.089             0.611                    0.282
J03WN1  target_aug      2         0.340 

## 3. Sanity checks on a real IDSSE match
* `target_like` ignores labels (shuffled labels give the same ball);
* both transforms commute with the home/away + 180° mirror (frames that differ are counted; float rounding at the touchline is the only expected source);
* `target_aug` is reproducible from its seed.

In [4]:
from dataclasses import replace
name = C.MATCH_IDS[0]
match = C1.load_degraded(name, 0)
shuffled = replace(match, labels=np.random.default_rng(1).permutation(match.labels))
same = lambda a, b: np.array_equal(a.ball, b.ball, equal_nan=True)
print('target_like label independence:', same(V.target_like(match), V.target_like(shuffled)))
for label, apply in (('target_like', V.target_like),
                     ('target_aug', lambda m: V.target_aug(m, np.random.default_rng(V.aug_seed(name, 1))))):
    a, b = apply(V.mirror_match(match)), V.mirror_match(apply(match))
    differ = ~((np.isnan(a.ball) & np.isnan(b.ball)) | np.isclose(a.ball, b.ball)).all(axis=1)
    print(f'{label} mirror commutation: frames that differ = {int(differ.sum())} of {len(differ)}')
r1 = V.target_aug(C1.load_degraded(name, 1), np.random.default_rng(V.aug_seed(name, 1)))
r2 = V.target_aug(C1.load_degraded(name, 1), np.random.default_rng(V.aug_seed(name, 1)))
print('target_aug reproducible:', same(r1, r2))
print('exp_2 fits cached so far:', P.cached_fit_count(), '| fingerprints:', {a: P.fingerprint(a) for a in P.MODEL_ARMS})

target_like label independence: True


target_like mirror commutation: frames that differ = 0 of 145968


target_aug mirror commutation: frames that differ = 0 of 145968


target_aug reproducible: True
exp_2 fits cached so far: 0 | fingerprints: {'exp1_ref': '3f61fc0f3305', 'exp2_aug': '5cea8203869e', 'exp2_aug_s1': '01776e3c0b77', 'exp2_aug_s2': 'fb137265b072'}
